# Program 1 — custom objects (YOLO11n-seg)

Merges all your own datasets, trains, cross-labels each dataset with the other objects
(you approve them by hand), then retrains. The result is the custom model that Program 2 uses.

**Google Drive layout** (you create `yolo` and `yolo/datasets` yourself):
```
MyDrive/yolo/
  datasets/            <- one zip per object (Roboflow YOLO segmentation export, one class each)
  custom/              <- created by this notebook: v1_2objects_2026-11-08/, v2_.../
  full/                <- created by Program 2
```
**Run:** Runtime → Change runtime type → T4 GPU, then Runtime → Run all.
It stops on purpose when you need to review labels. After a disconnect, just Run all again:
finished steps are skipped and training resumes.

In [ ]:
# ============================== SETTINGS ==============================
ROOT = "/content/drive/MyDrive/yolo"

# One line per object: (class name, zip file in ROOT/datasets).
# The order sets the class numbers (0, 1, 2 ...). Add new objects at the END only.
DATASETS = [
    ("stairs",         "stairs.zip"),
    ("tactile_paving", "tactile_paving.zip"),
]

VERSION = "auto"          # "auto": continue the newest unfinished version, else start the next vN
REVIEW = "manual"         # "manual": you delete wrong previews in Drive | "auto": accept all
REVIEWED_VERSION = ""     # after reviewing, paste the version name here (e.g. "v1_2objects_2026-11-08")

BOX_LABELS = "convert"    # box-only labels: "convert" to rectangles | "skip_image" | "stop" (list them)
CONF = 0.35               # cross-label confidence (lower = more proposals to review)
MIN_AREA = 0.002          # ignore proposed masks smaller than 0.2% of the image
VAL_FRACTION = 0.15       # used only if a dataset has no valid/ split

MODEL = "yolo11n-seg.pt"
PASS1_EPOCHS = 60         # first training, used to find the other objects in each dataset
PASS2_EPOCHS = 40         # retraining with the approved cross-labels
PASS2_START = "pass1"     # "pass1": continue from pass-1 model (faster) | "fresh": from MODEL
PATIENCE = 15
IMGSZ = 640
MAX_SIDE = 640            # photos are shrunk to this once (must be >= IMGSZ)
BATCH = 32
WORKERS = 2
TIMEZONE = "Asia/Singapore"
# =====================================================================

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!pip install -q -U ultralytics
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
# ---------------------------------------------------------------------------
# Shared helpers (this cell is identical in both notebooks)
# ---------------------------------------------------------------------------
import os, re, json, shutil, random, subprocess, datetime
from pathlib import Path
from zoneinfo import ZoneInfo
import numpy as np

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
LOCAL = Path("/content/work")        # fast local disk (lost on disconnect)
DATA = LOCAL / "data"                # dataset used for training


class StopHere(Exception):
    """Raised on purpose to stop 'Run all' and ask you to do something."""


def today():
    return datetime.datetime.now(ZoneInfo(TIMEZONE)).strftime("%Y-%m-%d")


def load_state(vdir):
    f = Path(vdir) / "state.json"
    return json.loads(f.read_text()) if f.exists() else {}


def save_state(vdir, st):
    Path(vdir).mkdir(parents=True, exist_ok=True)
    (Path(vdir) / "state.json").write_text(json.dumps(st, indent=1))


def version_dirs(parent):
    out = {}
    if Path(parent).is_dir():
        for d in Path(parent).iterdir():
            m = re.match(r"v(\d+)_", d.name)
            if d.is_dir() and m:
                out[int(m.group(1))] = d
    return out


def resolve_version(parent, version, n_objects):
    """'auto' = continue the newest unfinished version, otherwise start the next number."""
    found = version_dirs(parent)
    if version != "auto":
        return Path(parent) / version
    if found:
        newest = found[max(found)]
        if not load_state(newest).get("done"):
            return newest
    return Path(parent) / f"v{max(found, default=0) + 1}_{n_objects}objects_{today()}"


def run(cmd, cwd=None):
    subprocess.run(cmd, cwd=cwd, check=True)


def zip_dirs(zip_path, members, cwd):
    """Store-only zip of `members` (paths relative to cwd) into zip_path on Drive."""
    zip_path = Path(zip_path)
    tmp = LOCAL / ("_tmp_" + zip_path.name)
    tmp.unlink(missing_ok=True)
    run(["zip", "-q", "-r", "-0", str(tmp)] + [str(m) for m in members], cwd=cwd)
    shutil.copy(tmp, zip_path)
    tmp.unlink()


def unzip(zip_path, dest):
    Path(dest).mkdir(parents=True, exist_ok=True)
    run(["unzip", "-q", "-o", str(zip_path), "-d", str(dest)])


def images_in(split=None, prefix=None):
    splits = [split] if split else ["train", "val", "test"]
    out = []
    for s in splits:
        d = DATA / s / "images"
        if d.exists():
            out += [p for p in sorted(d.iterdir())
                    if p.suffix.lower() in IMG_EXTS and (prefix is None or p.name.startswith(prefix))]
    return out


def write_yaml(names):
    lines = [f"path: {DATA}", "train: train/images", "val: val/images"]
    if images_in("test"):
        lines.append("test: test/images")
    lines += [f"nc: {len(names)}", "names:"] + [f"  {i}: {n}" for i, n in enumerate(names)]
    (DATA / "data.yaml").write_text("\n".join(lines) + "\n")


def polygon_area(xy):
    x, y = xy[:, 0], xy[:, 1]
    return 0.5 * abs(float(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1))))


def propose(weights, images, keep, remap, conf, tag, review_dir):
    """Run `weights` on `images`, keep classes `keep`, write proposals + previews."""
    import cv2
    from ultralytics import YOLO
    pdir = Path(review_dir) / "proposals" / tag
    vdir = Path(review_dir) / "previews" / tag
    pdir.mkdir(parents=True, exist_ok=True)
    vdir.mkdir(parents=True, exist_ok=True)
    model = YOLO(str(weights))
    n = 0
    paths = [str(p) for p in images]
    for i in range(0, len(paths), 256):
        for r in model.predict(paths[i:i + 256], classes=keep, conf=conf, imgsz=IMGSZ,
                               retina_masks=True, stream=True, verbose=False, device=0):
            if r.masks is None or len(r.masks) == 0:
                continue
            lines = []
            for poly, c in zip(r.masks.xyn, r.boxes.cls.int().tolist()):
                if len(poly) < 3 or polygon_area(poly) < MIN_AREA:
                    continue
                pts = poly[::max(1, len(poly) // 150)].clip(0, 1).reshape(-1)
                lines.append(f"{remap[c]} " + " ".join(f"{v:.5f}" for v in pts))
            if not lines:
                continue
            ip = Path(r.path)
            key = f"{ip.parent.parent.name}__{ip.stem}"          # split__stem
            (pdir / f"{key}.txt").write_text("\n".join(lines) + "\n")
            cv2.imwrite(str(vdir / f"{key}.jpg"), r.plot())
            n += 1
    print(f"  {tag}: {n} images with proposals")
    return n


def apply_reviews(review_dir, manual):
    """Add approved proposals to the labels. Manual: a proposal counts only if its preview still exists."""
    added = rejected = 0
    for p in sorted((Path(review_dir) / "proposals").rglob("*.txt")):
        preview = Path(review_dir) / "previews" / p.parent.name / (p.stem + ".jpg")
        if manual and not preview.exists():
            rejected += 1
            continue
        split, stem = p.stem.split("__", 1)
        lbl = DATA / split / "labels" / f"{stem}.txt"
        old = lbl.read_text().rstrip("\n") if lbl.exists() else ""
        new = p.read_text().rstrip("\n")
        lbl.write_text((old + "\n" if old else "") + new + "\n")
        added += 1
    for c in DATA.rglob("*.cache"):
        c.unlink()
    print(f"Applied {added} images, rejected {rejected}")
    return added, rejected


def train_stage(vdir, st, name, start_weights, epochs):
    """Train (or resume after a disconnect). Results go to <version>/runs/<name> on Drive."""
    from ultralytics import YOLO
    run_dir = Path(vdir) / "runs" / name
    best, last = run_dir / "weights" / "best.pt", run_dir / "weights" / "last.pt"
    if st.get(f"{name}_done"):
        print(f"{name}: already trained")
        return best
    for c in DATA.rglob("*.cache"):
        c.unlink()
    if last.exists():
        print(f"{name}: resuming from {last}")
        try:
            YOLO(str(last)).train(resume=True)
        except Exception as e:      # e.g. it had already finished before the disconnect
            print("  resume not possible:", e)
    else:
        print(f"{name}: training from {start_weights} for up to {epochs} epochs")
        YOLO(str(start_weights)).train(
            data=str(DATA / "data.yaml"), task="segment", epochs=epochs, patience=PATIENCE,
            imgsz=IMGSZ, batch=BATCH, workers=WORKERS, amp=True, cos_lr=True, close_mosaic=10,
            device=0, project=str(Path(vdir) / "runs"), name=name, exist_ok=True, plots=True, seed=0)
    if not best.exists():
        raise RuntimeError(f"{name}: no best.pt produced, check the log above")
    st[f"{name}_done"] = True
    save_state(vdir, st)
    return best


def finish(vdir, st, best, names, extra):
    from ultralytics import YOLO
    vdir = Path(vdir)
    final = vdir / f"{vdir.name}.pt"
    shutil.copy(best, final)
    run_dir = Path(best).parent.parent
    for f in ["results.png", "results.csv", "confusion_matrix_normalized.png", "MaskPR_curve.png"]:
        if (run_dir / f).exists():
            shutil.copy(run_dir / f, vdir / f)
    info = {"version": vdir.name, "classes": names, "created": st.get("created"), **extra}
    (vdir / "info.json").write_text(json.dumps(info, indent=1))
    print("\nFinal check on the validation set:")
    YOLO(str(final)).val(data=str(DATA / "data.yaml"), imgsz=IMGSZ, batch=16, device=0,
                         project=str(LOCAL / "val"), name=vdir.name, exist_ok=True)
    st["done"] = True
    save_state(vdir, st)
    print(f"\nDONE. Model: {final}")

In [ ]:
ROOT_P = Path(ROOT)
DS_DIR = ROOT_P / "datasets"
assert ROOT_P.is_dir(), f"Create the folder {ROOT} in Google Drive first"
assert DS_DIR.is_dir(), f"Create the folder {DS_DIR} and put your dataset zips in it"
CLASSES = [n for n, _ in DATASETS]
assert len(set(CLASSES)) == len(CLASSES), "Two datasets have the same class name"
missing = [z for _, z in DATASETS if not (DS_DIR / z).exists()]
assert not missing, f"Not found in {DS_DIR}: {missing}"
assert MAX_SIDE >= IMGSZ, "MAX_SIDE must be >= IMGSZ"

VDIR = resolve_version(ROOT_P / "custom", VERSION, len(CLASSES))
VNAME = VDIR.name
REVIEW_DIR = VDIR / "review"
st = load_state(VDIR)
if not st:
    st = {"created": today(), "classes": CLASSES}
    save_state(VDIR, st)
elif st["classes"] != CLASSES:
    raise StopHere(f"{VNAME} was started with classes {st['classes']}, but DATASETS now lists {CLASSES}.\n"
                   f"Either put DATASETS back, or delete/rename the unfinished folder {VDIR} in Drive.")
print("Version:", VNAME, "| classes:", CLASSES)

if st.get("built") and not (DATA / "data.yaml").exists():      # after a disconnect
    print("Restoring dataset from Drive ...")
    unzip(VDIR / "dataset.zip", LOCAL)
    if (VDIR / "labels_reviewed.zip").exists():
        unzip(VDIR / "labels_reviewed.zip", LOCAL)

## 1. Build the merged dataset

In [ ]:
import cv2, yaml
if not st.get("built"):
    shutil.rmtree(LOCAL, ignore_errors=True)
    for s in ["train", "val", "test"]:
        (DATA / s / "images").mkdir(parents=True)
        (DATA / s / "labels").mkdir(parents=True)
    random.seed(0)
    stats, box_files = {}, []
    for k, (name, zname) in enumerate(DATASETS):
        src = LOCAL / "src" / name
        unzip(DS_DIR / zname, src)
        yamls = sorted(src.rglob("data.yaml"))
        assert yamls, f"{zname}: no data.yaml inside"
        root = yamls[0].parent
        nc = (yaml.safe_load(yamls[0].read_text()) or {}).get("nc", 1)
        if nc != 1:
            print(f"WARNING {zname}: has nc={nc}; every object in it will be treated as '{name}'")
        entries = []
        for s_src, s_dst in [("train", "train"), ("valid", "val"), ("val", "val"), ("test", "test")]:
            idir = root / s_src / "images"
            if idir.is_dir():
                entries += [[p, root / s_src / "labels" / f"{p.stem}.txt", s_dst]
                            for p in sorted(idir.iterdir()) if p.suffix.lower() in IMG_EXTS]
        if not any(e[2] == "val" for e in entries):
            tr = [e for e in entries if e[2] == "train"]
            for e in random.sample(tr, max(1, int(len(tr) * VAL_FRACTION))):
                e[2] = "val"
            print(f"{name}: no valid/ split, moved {VAL_FRACTION:.0%} of train to val")

        c = {"train": 0, "val": 0, "test": 0, "objects": 0, "box_only": 0, "no_label": 0, "skipped": 0}
        for img, lbl, split in entries:
            lines, has_box = [], False
            if lbl.exists():
                for raw in lbl.read_text().splitlines():
                    parts = raw.split()
                    if len(parts) < 5:
                        continue
                    v = [min(max(float(x), 0.0), 1.0) for x in parts[1:]]
                    if len(v) == 4:
                        has_box = True
                        xc, yc, w, h = v
                        v = [xc-w/2, yc-h/2, xc+w/2, yc-h/2, xc+w/2, yc+h/2, xc-w/2, yc+h/2]
                    if len(v) >= 6 and len(v) % 2 == 0:
                        lines.append(f"{k} " + " ".join(f"{x:.5f}" for x in v))
            else:
                c["no_label"] += 1
            if has_box:
                c["box_only"] += 1
                box_files.append(f"{zname}: {split}/{img.name}")
                if BOX_LABELS == "skip_image":
                    c["skipped"] += 1
                    continue
            im = cv2.imread(str(img))
            if im is None:
                c["skipped"] += 1
                continue
            h, w = im.shape[:2]
            r = MAX_SIDE / max(h, w)
            if r < 1:
                im = cv2.resize(im, (round(w*r), round(h*r)), interpolation=cv2.INTER_AREA)
            stem = f"{name}__{img.stem}"
            cv2.imwrite(str(DATA / split / "images" / f"{stem}.jpg"), im, [cv2.IMWRITE_JPEG_QUALITY, 92])
            (DATA / split / "labels" / f"{stem}.txt").write_text("\n".join(lines) + ("\n" if lines else ""))
            c[split] += 1
            c["objects"] += len(lines)
        stats[name] = c

    print(f"\n{'class':<20}" + "".join(f"{k:>10}" for k in next(iter(stats.values()))))
    for name, c in stats.items():
        print(f"{name:<20}" + "".join(f"{v:>10}" for v in c.values()))

    if BOX_LABELS == "stop" and box_files:
        (VDIR / "box_only_images.txt").write_text("\n".join(box_files) + "\n")
        raise StopHere(f"{len(box_files)} images have box-only labels, listed in {VDIR / 'box_only_images.txt'}.\n"
                       f"Redraw them as polygons, or set BOX_LABELS to 'convert' or 'skip_image', then Run all.")

    write_yaml(CLASSES)
    shutil.rmtree(LOCAL / "src")
    print("\nSaving dataset to Drive ...")
    zip_dirs(VDIR / "dataset.zip", ["data"], cwd=LOCAL)
    st.update(built=True, image_counts=stats)
    save_state(VDIR, st)
else:
    print("Dataset already built")

## 2. Pass 1 training

In [ ]:
best1 = train_stage(VDIR, st, "pass1", MODEL, PASS1_EPOCHS)

## 3. Cross-label: find the other objects in each dataset

In [ ]:
if len(CLASSES) < 2:
    print("Only one object: nothing to cross-label")
elif not st.get("proposed"):
    shutil.rmtree(REVIEW_DIR, ignore_errors=True)
    total = 0
    for k, name in enumerate(CLASSES):
        others = [i for i in range(len(CLASSES)) if i != k]
        total += propose(best1, images_in(prefix=f"{name}__"), others, {i: i for i in others},
                         CONF, f"in_{name}", REVIEW_DIR)
    st.update(proposed=True, n_proposals=total)
    save_state(VDIR, st)
else:
    print(f"Proposals already made: {st['n_proposals']}")

## 4. Review and apply
With `REVIEW = "manual"` the run stops here the first time. In Google Drive open
`yolo/custom/<version>/review/previews/`. Each sub-folder (`in_stairs`, `in_tactile_paving`, …) shows
images where the model found *other* objects. **Delete every preview that is wrong.**
Then set `REVIEWED_VERSION` in the settings to the version name and Runtime → Run all.

In [ ]:
if len(CLASSES) >= 2 and not st.get("applied"):
    n = st.get("n_proposals", 0)
    if n and REVIEW == "manual" and REVIEWED_VERSION != VNAME:
        raise StopHere(f"""
REVIEW NEEDED: {n} images have proposed labels.
  1. In Google Drive open: {REVIEW_DIR / 'previews'}
  2. Delete every preview image whose new outline is wrong (kept = approved).
  3. In the SETTINGS cell set:  REVIEWED_VERSION = "{VNAME}"
  4. Runtime -> Run all""")
    added = apply_reviews(REVIEW_DIR, manual=(REVIEW == "manual"))[0] if n else 0
    zip_dirs(VDIR / "labels_reviewed.zip",
             ["data/train/labels", "data/val/labels", "data/test/labels"], cwd=LOCAL)
    st.update(applied=True, n_applied=added)
    save_state(VDIR, st)
elif st.get("applied"):
    print(f"Already applied: {st.get('n_applied', 0)} images")

## 5. Pass 2 training and final model

In [ ]:
if st.get("done"):
    print("This version is finished:", VDIR / f"{VNAME}.pt")
else:
    if len(CLASSES) >= 2 and st.get("n_applied", 0) > 0:
        start = best1 if PASS2_START == "pass1" else MODEL
        best = train_stage(VDIR, st, "pass2", start, PASS2_EPOCHS)
    else:
        print("No cross-labels added, so the pass-1 model is the final model")
        best = best1
    finish(VDIR, st, best, CLASSES, {"image_counts": st["image_counts"]})